In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD DATASETS FROM COLAB WORKSPACE
# ==========================================
print("Loading datasets...")

# --- CRITICAL FIX: skip the title banner and look at row 2 (index 1) or row 3 (index 2) for headers ---
try:
    df_bd = pd.read_excel('Book2.xlsx', header=1)
    # If 'age' still isn't found, we try shifting down one more row:
    if 'age' not in [str(c).lower().strip() for c in df_bd.columns] and 'unnamed' in str(df_bd.columns[1]):
        df_bd = pd.read_excel('Book2.xlsx', header=2)
except Exception as e:
    df_bd = pd.read_excel('Book2.xlsx')

df_anemia = pd.read_csv('anemia.csv')
df_obesity = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')

# Clean and Lowercase EVERY column name
df_bd.columns = df_bd.columns.str.strip().str.lower()
df_anemia.columns = df_anemia.columns.str.strip().str.lower()
df_obesity.columns = df_obesity.columns.str.strip().str.lower()

# Print out your updated column names to verify we successfully broke past the banner!
print("Updated Book2 columns:", df_bd.columns.tolist())

# ==========================================
# 2. STANDARDIZE THE PRIMARY BACKBONE DATASET (Bangladeshi Clinical Data)
# ==========================================
print("Standardizing primary clinical features...")
bd_clean = pd.DataFrame()

# 1. Demographics
bd_clean['age'] = df_bd['age'].astype(float)

# 2. FIX: Safely extract numbers from pregnancy week text (e.g., '38 week' -> 38.0)
week_col = [c for c in df_bd.columns if 'week' in c or 'gestational' in c]
if week_col:
    # str.extract(r'(\d+)') grabs only the numeric digits from the text string
    bd_clean['pregnancy_week'] = df_bd[week_col[0]].astype(str).str.extract(r'(\d+)').astype(float)
    # Fill any missing values with a safe median pregnancy baseline (20 weeks)
    bd_clean['pregnancy_week'] = bd_clean['pregnancy_week'].fillna(20.0)
else:
    bd_clean['pregnancy_week'] = 20.0

# 3. FIX: Safely extract numbers from weight just in case it contains text like '65 kg'
weight_col = [c for c in df_bd.columns if 'weight' in c or 'ওজন' in c]
if weight_col:
    bd_clean['weight'] = df_bd[weight_col[0]].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
    bd_clean['weight'] = bd_clean['weight'].fillna(60.0)
else:
    bd_clean['weight'] = 60.0

# 4. Custom parser to convert Foot.Inches formatting securely to pure meters
def parse_height_to_meters(h_val):
    try:
        val_str = str(h_val).strip()
        if '.' in val_str:
            parts = val_str.split('.')
            feet = int(parts[0])
            inches = int(parts[1]) if parts[1] else 0
        else:
            feet = int(h_val)
            inches = 0
        return ((feet * 12) + inches) * 0.0254
    except:
        return 1.55

height_col = [c for c in df_bd.columns if 'height' in c or 'উচ্চতা' in c]
height_m = df_bd[height_col[0]].apply(parse_height_to_meters) if height_col else 1.55
bd_clean['bmi'] = bd_clean['weight'] / (height_m ** 2)

# 5. Vectorizing raw Blood Pressure strings
bp_col = [c for c in df_bd.columns if 'pressure' in c or 'bp' in c or 'চাপ' in c]
df_bd['bp_temp'] = df_bd[bp_col[0]].astype(str).str.replace(' ', '')
bd_clean['systolic_bp'] = df_bd['bp_temp'].str.split('/', expand=True)[0].astype(float)
bd_clean['diastolic_bp'] = df_bd['bp_temp'].str.split('/', expand=True)[1].astype(float)

# 6. Clean up clinical text validations
def text_to_binary(val):
    v = str(val).lower()
    return 1 if 'pos' in v or 'yes' in v or 'আছে' in v or 'হ্যাঁ' in v else 0

sugar_col = [c for c in df_bd.columns if 'sugar' in c]
bd_clean['urine_sugar'] = df_bd[sugar_col[0]].apply(text_to_binary) if sugar_col else 0

albumin_col = [c for c in df_bd.columns if 'albumin' in c]
bd_clean['urine_albumin'] = df_bd[albumin_col[0]].apply(text_to_binary) if albumin_col else 0

# 7. Setup Target 1: Overall Risk
risk_col = [c for c in df_bd.columns if 'risk' in c or 'ঝুঁকি' in c]
bd_clean['target_risk'] = df_bd[risk_col[0]].apply(lambda x: 2 if 'yes' in str(x).lower() or 'হ্যাঁ' in str(x) or 'high' in str(x).lower() else 0) if risk_col else 0

# Sort strictly by age to create an ordered index for demographic stitching
bd_clean = bd_clean.sort_values(by='age').reset_index(drop=True)
print("Primary clinical data cleaned and parsed successfully!")

Loading datasets...
Updated Book2 columns: ['name', 'age', 'gravida', 'titi tika', 'pregnancy_week', 'weight', 'height', 'raw_bp', 'raw_anemia', 'জন্ডিস', 'গর্ভস্হ শিশু অবস্থান', 'গর্ভস্হ শিশু নাড়াচাড়া', 'গর্ভস্হ শিশু হৃৎস্পন্দন', 'urine_albumin', 'urine_sugar', 'vdrl', 'hrsag', 'raw_high_risk']
Standardizing primary clinical features...
Primary clinical data cleaned and parsed successfully!


In [ ]:
# ==========================================
# 3. PREPROCESS AND EXTRACT TARGET 2 (Anemia Metrics)
# ==========================================
print("Extracting anemia profiles...")

# Clean and lowercase columns one more time locally just to be 100% safe
df_anemia.columns = df_anemia.columns.str.strip().str.lower()

# Dynamic detection for gender/sex column in anemia dataset
gender_col_anemia = [c for c in df_anemia.columns if 'gender' in c or 'sex' in c]
if not gender_col_anemia:
    raise KeyError(f"Could not find gender column in anemia.csv. Available columns: {df_anemia.columns.tolist()}")

# Filter for females. In this dataset, Gender 1 represents Female.
df_anemia_fem = df_anemia[
    (df_anemia[gender_col_anemia[0]] == 1) |
    (df_anemia[gender_col_anemia[0]].astype(str).str.lower().str.startswith('f'))
].copy().reset_index(drop=True)

# FIX: Since this specific anemia.csv has no age column, we skip age sorting/filtering for it!
print(f"Found {len(df_anemia_fem)} female anemia records. Proceeding with direct index mapping.")


# ==========================================
# 4. PREPROCESS AND EXTRACT EXTRA NUTRITIONAL LIFESTYLE METRICS
# ==========================================
print("Extracting lifestyle nutrition distributions...")

# Clean and lowercase obesity dataset columns locally
df_obesity.columns = df_obesity.columns.str.strip().str.lower()

# Dynamic detection for gender/sex column in obesity dataset
gender_col_obesity = [c for c in df_obesity.columns if 'gender' in c or 'sex' in c]
df_obesity_fem = df_obesity[df_obesity[gender_col_obesity[0]].astype(str).str.lower() == 'female'].copy()

# Dynamic detection for age column in obesity dataset
age_col_obesity = [c for c in df_obesity_fem.columns if 'age' in c]

# This dataset DOES have age, so we safely filter childbearing years here
df_obesity_fem = df_obesity_fem[(df_obesity_fem[age_col_obesity[0]] >= 15) & (df_obesity_fem[age_col_obesity[0]] <= 45)]
df_obesity_fem = df_obesity_fem.sort_values(by=age_col_obesity[0]).reset_index(drop=True)


# ==========================================
# 5. DEMOGRAPHIC STITCHING MATRIX (Blending them together)
# ==========================================
print("Merging data streams via aligned indices...")
# Bind rows up to the minimum available across files to preserve structured arrays
target_row_limit = min(len(bd_clean), len(df_anemia_fem), len(df_obesity_fem))

master_df = bd_clean.iloc[:target_row_limit].copy()

# Dynamic detection for hemoglobin column in anemia dataset
hemo_col = [c for c in df_anemia_fem.columns if 'hemoglobin' in c or 'hgb' in c]
master_df['hemoglobin'] = df_anemia_fem[hemo_col[0]].iloc[:target_row_limit].values

# Dynamic detection for dietary lifestyle features from the lifestyle data
fcvc_col = [c for c in df_obesity_fem.columns if 'fcvc' in c or 'veg' in c]
ncp_col = [c for c in df_obesity_fem.columns if 'ncp' in c or 'meal' in c]

master_df['veg_freq'] = df_obesity_fem[fcvc_col[0]].iloc[:target_row_limit].round().astype(int).values
master_df['meals_per_day'] = df_obesity_fem[ncp_col[0]].iloc[:target_row_limit].round().astype(int).values


# ==========================================
# 6. CALCULATING COMPLEX MULTI-OUTPUT TARGET ARRAYS (Y)
# ==========================================
print("Mapping target output vectors...")

def calculate_nutritional_deficit(row):
    if row['hemoglobin'] < 11.0:
        return 1  # Iron Deficit / Anemia
    elif row['meals_per_day'] <= 2 or row['bmi'] < 18.5:
        return 2  # Caloric/Protein Deficit
    return 0      # Normal

master_df['target_nutrition'] = master_df.apply(calculate_nutritional_deficit, axis=1)

def parse_acute_clinical_anomalies(row):
    if row['systolic_bp'] >= 145 or row['diastolic_bp'] >= 95 or row['urine_albumin'] == 1:
        return 1  # Critical Warning Alert
    return 0

master_df['target_anomaly'] = master_df.apply(parse_acute_clinical_anomalies, axis=1)

# Ensure backup default for blood sugar feature
master_df['blood_sugar'] = 5.2

final_features_list = [
    'age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'hemoglobin',
    'bmi', 'meals_per_day', 'veg_freq',
    'target_risk', 'target_nutrition', 'target_anomaly'
]

final_matrix = master_df[final_features_list].dropna()

# Save final clean csv output
final_matrix.to_csv('maatri_master_dataset.csv', index=False)

print("\n=== PREPROCESSING COMPLETED SUCCESSFULLY ===")
print(f"Total unified patient vectors compiled: {len(final_matrix)}")
print("Your production-ready training data is saved in your Colab files as: 'maatri_master_dataset.csv'")

Extracting anemia profiles...
Found 740 female anemia records. Proceeding with direct index mapping.
Extracting lifestyle nutrition distributions...
Merging data streams via aligned indices...
Mapping target output vectors...

=== PREPROCESSING COMPLETED SUCCESSFULLY ===
Total unified patient vectors compiled: 740
Your production-ready training data is saved in your Colab files as: 'maatri_master_dataset.csv'


In [ ]:
df_master = pd.read_csv('maatri_master_dataset.csv')
df_master.head()

,age,systolic_bp,diastolic_bp,blood_sugar,hemoglobin,bmi,meals_per_day,veg_freq,target_risk,target_nutrition,target_anomaly
0,18.0,100.0,60.0,5.2,14.9,20.811655,3,3,2,0,0
1,18.0,90.0,60.0,5.2,14.7,22.060354,1,2,2,2,0
2,18.0,120.0,60.0,5.2,12.7,20.811655,4,2,2,0,0
3,18.0,100.0,70.0,5.2,12.7,24.141519,1,2,0,2,0
4,18.0,100.0,55.0,5.2,14.9,25.806452,3,3,0,0,0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_master)

https://docs.google.com/spreadsheets/d/1U2iV9V32FVm7iqZRQu6xXdrrj_vYESyxcC9JBufC620/edit#gid=0


In [ ]:
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split

# 1. Load the compiled dataset
df_master = pd.read_csv('maatri_master_dataset.csv')

# --- FIX 1: Remap risk classes from [0, 2] to consecutive integers [0, 1] ---
df_master['target_risk'] = df_master['target_risk'].replace({2: 1})

# --- FIX 2: Re-calculate anomaly triggers to fit this specific dataset's range ---
def check_for_acute_anomaly(row):
    if row['systolic_bp'] >= 115 or row['diastolic_bp'] >= 75:
        return 1
    return 0

df_master['target_anomaly'] = df_master.apply(check_for_acute_anomaly, axis=1)

# 2. Isolate Input Features (X)
X = df_master[['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'hemoglobin', 'bmi', 'meals_per_day', 'veg_freq']]

# 3. Isolate Target Arrays (Y)
Y = df_master[['target_risk', 'target_nutrition', 'target_anomaly']]

# 4. Generate Train/Test Splits
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 5. Initialize and Train the Single Unified Architecture
# NOTE: Removing the forced multi_class hyperparameter allows MultiOutputClassifier
# to automatically optimize binary vs multi-class targets for each column independently!
base_estimator = XGBClassifier(random_state=42, eval_metric='logloss')
unified_brain = MultiOutputClassifier(base_estimator)

print("Training your multi-output model engine...")
unified_brain.fit(X_train, y_train)

# 6. Performance Evaluation
train_acc = unified_brain.score(X_train, y_train)
test_acc = unified_brain.score(X_test, y_test)

print(f"\nTraining Set Accuracy: {train_acc * 100:.2f}%")
print(f"Validation Testing Set Accuracy: {test_acc * 100:.2f}%")

# 7. EXPORT FOR DEPLOYMENT TO YOUR FASTAPI BACKEND
model_output_name = 'maatri_unified_model.pkl'
joblib.dump(unified_brain, model_output_name)

print(f"\nSuccess! '{model_output_name}' generated cleanly. Download it from your sidebar and drop it in Render!")

Training your multi-output model engine...

Training Set Accuracy: 100.00%
Validation Testing Set Accuracy: 89.86%

Success! 'maatri_unified_model.pkl' generated cleanly. Download it from your sidebar and drop it in Render!


In [ ]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split

# 1. Load your compiled dataset
df_master = pd.read_csv('maatri_master_dataset.csv')

# ==========================================
# RE-ENGINEERING THREE-TIER RISK LEVELS (0, 1, 2)
# ==========================================
def inject_three_tier_boundaries(row):
    # --- CRITICAL HIGH RISK BOUNDARIES (Level 2) ---
    if row['systolic_bp'] >= 140 or row['diastolic_bp'] >= 90:
        return pd.Series([2, 1]) # [target_risk=2, target_anomaly=1]

    # --- MODERATE / MEDIUM RISK BOUNDARIES (Level 1) ---
    elif (120 < row['systolic_bp'] < 140) or (80 < row['diastolic_bp'] < 90) or (row['veg_freq'] == 1):
        return pd.Series([1, 0]) # [target_risk=1, target_anomaly=0]

    # --- PERFECTLY SAFE / LOW RISK BOUNDARIES (Level 0) ---
    else:
        return pd.Series([0, 0]) # [target_risk=0, target_anomaly=0]

# Overwrite risk and anomaly tables with clean mathematical thresholds
df_master[['target_risk', 'target_anomaly']] = df_master.apply(inject_three_tier_boundaries, axis=1)

# --- GENERATE SYNTHETIC HIGH-RISK CASES (Level 2) ---
synthetic_high_risk_rows = []
for _ in range(200):
    synthetic_high_risk_rows.append({
        'age': np.random.randint(22, 38),
        'systolic_bp': np.random.randint(140, 165),
        'diastolic_bp': np.random.randint(90, 105),
        'blood_sugar': round(np.random.uniform(6.5, 9.5), 1),
        'hemoglobin': round(np.random.uniform(8.0, 10.5), 1),
        'bmi': round(np.random.uniform(17.5, 27.0), 1),
        'meals_per_day': np.random.choice([1, 2]),
        'veg_freq': 2,
        'target_risk': 2,        # High Risk
        'target_nutrition': 1,
        'target_anomaly': 1      # Emergency Alert
    })

# --- NEW: GENERATE SYNTHETIC MEDIUM-RISK CASES (Level 1) ---
# This provides the necessary examples for pre-hypertensive metrics
synthetic_med_risk_rows = []
for _ in range(200):
    synthetic_med_risk_rows.append({
        'age': np.random.randint(18, 35),
        'systolic_bp': np.random.randint(122, 138), # Pre-hypertension range
        'diastolic_bp': np.random.randint(82, 89),   # Pre-hypertension range
        'blood_sugar': round(np.random.uniform(5.0, 6.0), 1),
        'hemoglobin': round(np.random.uniform(10.5, 11.5), 1),
        'bmi': round(np.random.uniform(19.0, 24.0), 1),
        'meals_per_day': 3,
        'veg_freq': 2,
        'target_risk': 1,        # Medium Risk
        'target_nutrition': 0,
        'target_anomaly': 0      # No Acute Emergency
    })

df_synth_high = pd.DataFrame(synthetic_high_risk_rows)
df_synth_med = pd.DataFrame(synthetic_med_risk_rows)

# Combine everything into a fully representative multi-class distribution
df_final_training = pd.concat([df_master, df_synth_high, df_synth_med], ignore_index=True)

# ==========================================
# 2. RUN EXTRACTION AND INFERENCE SEPARATION
# ==========================================
X = df_final_training[['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'hemoglobin', 'bmi', 'meals_per_day', 'veg_freq']]
Y = df_final_training[['target_risk', 'target_nutrition', 'target_anomaly']]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

# ==========================================
# 3. TRAINING THE ENTIRE BRAIN
# ==========================================
base_estimator = XGBClassifier(max_depth=5, learning_rate=0.1, random_state=42, eval_metric='mlogloss')
unified_brain = MultiOutputClassifier(base_estimator)

print("Training your 3-Tier Multi-Output Model Engine...")
unified_brain.fit(X_train, y_train)

# ==========================================
# 4. RUN FINAL VERIFICATION TESTING SUITE
# ==========================================
print("\n=== RUNNING 3-TIER VERIFICATION TARGETS ===")

test_high = np.array([[24.0, 145.0, 95.0, 7.8, 9.2, 18.0, 2, 1]])   # High
test_med = np.array([[26.0, 128.0, 84.0, 5.2, 11.5, 20.0, 3, 2]])   # Medium
test_low = np.array([[28.0, 110.0, 70.0, 4.5, 12.5, 22.0, 3, 3]])   # Low

print("CRITICAL HIGH PROFILE -> Predicted Matrix:", unified_brain.predict(test_high), "(Expected Risk: 2)")
print("MEDIUM RISK PROFILE  -> Predicted Matrix:", unified_brain.predict(test_med), "(Expected Risk: 1)")
print("SAFE LOW PROFILE     -> Predicted Matrix:", unified_brain.predict(test_low), "(Expected Risk: 0)")

# Save the final functional pkl model binary
joblib.dump(unified_brain, 'maatri_unified_model.pkl')
print("\nSuccess! 3-tier 'maatri_unified_model.pkl' created and verified.")

Training your 3-Tier Multi-Output Model Engine...

=== RUNNING 3-TIER VERIFICATION TARGETS ===
CRITICAL HIGH PROFILE -> Predicted Matrix: [[2 1 1]] (Expected Risk: 2)
MEDIUM RISK PROFILE  -> Predicted Matrix: [[1 0 0]] (Expected Risk: 1)
SAFE LOW PROFILE     -> Predicted Matrix: [[0 0 0]] (Expected Risk: 0)

Success! 3-tier 'maatri_unified_model.pkl' created and verified.
